# Notebook 3: Inference Pipeline

**What:** How `apply_model` runs separation: chunking, overlap, shift trick.

**Why:** Long audio exceeds GPU memory. Chunking + overlap avoids boundary artifacts. Shift trick improves quality.

**How:** Load pretrained model, run separation on a short file, inspect the flow.

## 1. Load Pretrained Model

Use `mdx` or `htdemucs`. For 3070 Ti: add `segment=7` to reduce memory (htdemucs max ~7.8s).

In [ ]:
import sys
sys.path.insert(0, r'D:\demucs')

import torch
from demucs.pretrained import get_model
from demucs.apply import apply_model

# Choose model: htdemucs (default, best), mdx (older, lighter)
model_name = 'htdemucs'
model = get_model(model_name)
print(f"Model: {model_name}")
print(f"Sources: {model.sources}")
print(f"Segment (s): {getattr(model, 'segment', 'N/A')}")

# For 3070 Ti: override segment to use less VRAM
if hasattr(model, 'segment'):
    model.segment = 7.0  # seconds (htdemucs max ~7.8)
    print(f"Using segment=7 for 3070 Ti")

## 2. Prepare Input Audio

Create or load a short stereo wav. Demucs expects (batch, channels, samples) at 44.1kHz.

In [ ]:
import torchaudio
from pathlib import Path

# Option A: Use your own short file (e.g. 30 sec)
audio_path = Path("test_separation.wav")
if not audio_path.exists():
    # Option B: Generate test file
    sr = 44100
    duration = 10  # seconds
    mix = torch.randn(2, sr * duration) * 0.2
    torchaudio.save(str(audio_path), mix, sr)
    print("Created test_separation.wav (10s)")

wav, sr = torchaudio.load(str(audio_path))
mix = wav.unsqueeze(0)  # (1, 2, T)
print(f"Mix shape: {mix.shape}, sr={sr}")

## 3. Run Separation

`apply_model(model, mix, ...)` handles:
- Chunking into segments
- Overlap blending (default 0.25)
- Shift trick if `shifts > 1`
- Device placement

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

with torch.no_grad():
    sources = apply_model(
        model,
        mix.to(device),
        device=device,
        segment=7,      # override if needed for 3070 Ti
        overlap=0.25,
        shifts=1,       # use 2-5 for better quality (slower)
        split=True,     # chunk long audio
    )

print(f"Output shape: {sources.shape}  # (batch, sources, channels, samples)")

## 4. Save Stems

Save each source as wav file.

In [ ]:
out_dir = Path("separated_test")
out_dir.mkdir(exist_ok=True)

for i, name in enumerate(model.sources):
    stem = sources[0, i].cpu()  # (channels, samples)
    torchaudio.save(out_dir / f"{name}.wav", stem, sr)
    print(f"Saved {name}.wav")

print(f"Outputs in {out_dir.absolute()}")

## 5. Shift Trick (What & Why)

**What:** Run model multiple times with random circular shifts of input; average outputs.

**Why:** Convolutions are translation-equivariant, but chunk boundaries are not. Shifting + averaging reduces boundary artifacts.

**Cost:** `shifts`× slower. Use `shifts=2` for a trade-off on 3070 Ti.

In [ ]:
# Example: 2 shifts (2× time, better quality)
# sources = apply_model(model, mix, device=device, segment=7, shifts=2)

## 6. Command-Line Equivalent

```powershell
cd D:\demucs
python -m demucs --segment 7 -d cuda "path\to\song.mp3"
```

**Next:** Notebook 4 — Training on 3070 Ti.